In [166]:
%matplotlib inline

import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from ipywidgets import VBox, HBox, FloatSlider, Output, Label, Dropdown
from scipy.interpolate import RegularGridInterpolator
from IPython.display import display, clear_output, Audio
import sys
import PathGeneration as pg
from Interpolation import SplineTrajectory3D
import json
from matplotlib import ticker
from scipy.special import jv

build_dir = "../cpp/build"
if build_dir not in sys.path:
    sys.path.insert(0, build_dir)

import propagationmod
import resamplemod

In [167]:
# audio utils

def db_scale(x, amin=1e-10):
    return 20.0 * np.log10(np.maximum(x, amin))

def lin_scale(x):
    return 10**(x / 20)

def extend_noise(x, target_samples):
    if len(x) < target_samples: 
        X = np.fft.rfft(x)
        X *= np.exp(1j*np.random.rand(len(X))*2*np.pi)
        x = np.tile(np.fft.irfft(X), (target_samples-1)//len(x)+1)
    return x[:target_samples]

def extend_audio(x, target_samples):
    if len(x) >= target_samples:
        return x[: target_samples]
    fade = 2**11
    n = len(x)
    window = np.minimum(np.minimum(np.arange(n)/fade, n/fade-np.arange(n)/fade), 1)
    grain = x * (0.5 - 0.5 * np.cos(np.pi*window))
    y = np.zeros(((target_samples-1)//n+2)*n + 4 * fade)
    for i in range(0, len(y)-n-1, n-fade):
        y[i:i+n] += grain
    return y[fade:fade+target_samples]

def get_audio_from_source(source, mono=True):
    if isinstance(source, str):
        data, sr = sf.read(source, always_2d=True)
        if data.size == 0:
            raise RuntimeError("Empty or unsupported audio file.")
        if mono:
            audio = data[:, 0]
        return (audio, sr)
    elif isinstance(source, tuple):
        return source
    else:
        raise TypeError("bad scene audio type")
    
def propagate_audio(position_data, mic_positions, audio, sr):
    if isinstance(position_data, tuple):
        pos, vel = position_data
        return propagationmod.simulate_audio_propagation(sr, mic_positions, pos, vel, audio)
    else:
        return propagationmod.static_audio_propagation(sr, mic_positions, position_data, audio)

def rms_normalize(x):
    return x / (np.sqrt(np.mean(x * x)) + 1e-18)

def prepare_audio(audio_source, mic_array, position, total_samples, sr):
    
    audio, sr2 = get_audio_from_source(audio_source)
    
    if sr2 != sr:
        audio = resamplemod.sinc_resample(audio, sr/sr2, 16)
    
    audio = rms_normalize(audio)
    audio = extend_audio(audio, total_samples)
    audio = propagate_audio(position, mic_array, audio, sr)

    return audio

def create_uncorrelated_noise(m, n):
    return rms_normalize(np.random.randn(m, n))

def create_diffuse_noise(n, micpos, sr):
    m = micpos.shape[1]
    pos = np.random.randn(3, n)
    pos /= np.linalg.norm(pos, axis=0)
    noise = propagationmod.planewave_propagation(sr, 1.0, micpos, pos, np.random.randn(n))
    pos = np.flip(pos)
    noise += propagationmod.planewave_propagation(sr, 1.0, micpos, pos, np.random.randn(n))
    noise += np.roll(noise, noise.shape[1]//8)
    noise += np.roll(noise, noise.shape[1]//4)
    noise += np.roll(noise, noise.shape[1]//2)
    noise = rms_normalize(noise)
    return noise

In [168]:
# scene utils

todeg = 1 / np.pi * 180

class Scene:
    def __init__(self):
        self.target_position = ()
        self.interferer_position = []
        self.interferer_audio = []
        self.recorded_target = None     


def prepare_position(position, t):
    if isinstance(position, tuple):
        pos, t2 = position
        return SplineTrajectory3D(t2, pos).get(t)
    return position

def create_prepared_scene(scene, duration, sr):
    prepared_scene = Scene()
    t = np.linspace(0, duration, duration*sr)
    prepared_scene.target_position = prepare_position(scene.target_position, t)
    prepared_scene.interferer_position = [prepare_position(i, t) for i in scene.interferer_position]
    prepared_scene.interferer_audio = [get_audio_from_source(i) for i in scene.interferer_audio]
    return prepared_scene

def scene_center(duration, sr):
    scene = Scene()
    scene.target_position = np.array([0, 0, 10.0])
    return create_prepared_scene(scene, duration, sr)

def scene_stationary(duration, sr):
    scene = Scene()
    scene.target_position = np.array([2.0, 5.0, 5.0])
    return create_prepared_scene(scene, duration, sr)

def scene_line(duration, sr):
    scene = Scene()
    scene.target_position = pg.straight_line(
        3,
        duration,
        np.array([5, -5, 5]),
        np.array([5, 5, 5]))
    return create_prepared_scene(scene, duration, sr)

def scene_circle(duration, sr):
    scene = Scene()
    scene.target_position = pg.circular_orbit(
        100,
        duration,
        radius=5)
    return create_prepared_scene(scene, duration, sr)

def scene_circle2(duration, sr):
    scene = Scene()
    scene.target_position = pg.circular_orbit(
        100,
        duration,
        center=np.array([0.0, 0.0, 0.0]),
        radius = 7)
    return create_prepared_scene(scene, duration, sr)

def scene_helix(duration, sr):
    scene = Scene()
    scene.target_position = pg.helix_climb(
        100,
        duration,
        radius=3,
        height=9,
        turns = 2)
    return create_prepared_scene(scene, duration, sr)

def scene_random(duration, sr):
    scene = Scene()
    scene.target_position = pg.random_waypoints_spline(
        6,
        duration,
        np.array([-7, -7, 1]),
        np.array([7, 7, 10])
    )
    return create_prepared_scene(scene, duration, sr)

def scene_zeni(duration, sr):
    scene = Scene()
    t = np.linspace(0.0, duration, 200)
    theta = np.pi * (t / duration)

    x = 7 * np.cos(theta)
    y = np.full_like(x, 0)
    z = 7 * np.sin(theta)

    points = np.vstack((x, y, z))
    scene.target_position = (points, t)
    return create_prepared_scene(scene, duration, sr)

def scene_interferer(duration, sr):
    scene = Scene()
    scene.target_position = pg.circular_orbit(
        100,
        duration,
        radius=5)
    scene.interferer_position.append(np.array([2.0, 5.0, 5.0]))
    scene.interferer_audio.append((np.random.randn(duration*sr), sr))
    return create_prepared_scene(scene, duration, sr)

def get_position(position, t_idx):
    if isinstance(position, tuple):
        return position[0][:, t_idx]
    else:
        return position


In [169]:
# microphone arrays

class ArrayState:
    positions = np.array([[0, 0, 0]])
    cropac = {}
    calibration = None
    recorded_diffuse = None
    recorded_self_noise = None
    def __init__(self, pos=np.array([[0, 0, 0]]), cropac={}):
        self.positions = pos
        self.cropac = cropac
        

def square4():
    return ArrayState(np.array(
    [
        [-0.1, -0.1, 0],
        [-0.1,  0.1, 0],
        [ 0.1, -0.1, 0],
        [ 0.1,  0.1, 0],
    ]).T)

def square8():
    return ArrayState(np.array(
    [
        [-0.2, -0.2, 0],
        [-0.2,  0.2, 0],
        [ 0.2, -0.2, 0],
        [ 0.2,  0.2, 0],
        [-0.0, -0.2, 0],
        [-0.2,  0.0, 0],
        [ 0.2, -0.0, 0],
        [ 0.0,  0.2, 0],
    ]).T)

def square8_2():
    d = 0.06
    f = np.sqrt(2)
    yo = 0.00
    return ArrayState(np.array(
    [
        [-d*f, -d*f + yo, 0],
        [-d*f,  d*f + yo, 0],
        [ d*f, -d*f + yo, 0],
        [ d*f,  d*f + yo, 0],
        [-0.0, -d*2, 0],
        [-d*2,  0.0, 0],
        [ d*2, -0.0, 0],
        [ 0.0,  d*2, 0],
    ]).T)

def hexagon():
    d = 0.06
    a = []
    for i in range(6):
        a. append([np.cos(i*np.pi/3)*d, np.sin(i*np.pi/3)*d, 0])
    return ArrayState(np.array(a).T)

def random8d01():
    return ArrayState(np.stack((np.random.randn(8), np.random.randn(8), np.zeros(8))) * 0.1)

def random16d01():
    return ArrayState(np.stack((np.random.randn(16), np.random.randn(16), np.zeros(16))) * 0.1)

def cartesian_product(*arrays):
    """
    Return the Cartesian product of 1D NumPy arrays as a 2D array
    of shape (prod(len(ai)), len(arrays)).
    """
    if not arrays:
        return np.empty((0, 0))
    arrays = [np.asarray(a) for a in arrays]
    grids = np.meshgrid(*arrays, indexing='ij')        # one grid per input
    return np.stack(grids, axis=-1).reshape(-1, len(arrays))

def cube3():
    d = 0.01
    return ArrayState(cartesian_product(np.linspace(-d, d, 3), np.linspace(-d, d, 3), np.linspace(-d, d, 3)).T)

def linear8():
    return ArrayState(np.stack((np.linspace(0, 0.2, 8), np.zeros(8), np.zeros(8))))

def square9():
    return ArrayState(np.array(
    [
        [-0.2, -0.2, 0],
        [-0.2,  0.2, 0],
        [ 0.2, -0.2, 0],
        [ 0.2,  0.2, 0],
        [-0.0, -0.2, 0],
        [-0.2,  0.0, 0],
        [ 0.2, -0.0, 0],
        [ 0.0,  0.2, 0],
        [ 0.0,  0.0, 0],
    ]).T)

def load_array(path, calibrationPath=None, diffusePath=None, noisePath=None):

    array = ArrayState()
    with open(path, "r", encoding="utf-8") as f:
        array.__dict__ = json.load(f)
    array.positions = np.stack((*np.array(array.positions).T, np.zeros(len(array.positions))))
    
    if calibrationPath != None:
        with open(calibrationPath, "r", encoding="utf-8") as f:
            array.calibration = np.array(json.load(f))

    if diffusePath != None:
        array.recorded_diffuse = sf.read(diffusePath)

    if noisePath != None:
        array.recorded_diffuse = sf.read(noisePath)

    array.positions[0] -= np.mean(array.positions[0])
    array.positions[1] -= np.mean(array.positions[1])
    
    return array

def default8_1():
    return load_array("default_array8_1.json", "default_array8_calibration.json")

def default8_2():
    return load_array("default_array8_2.json", "default_array8_calibration.json")

def default8_3():
    return load_array("default_array8_3.json", "default_array8_calibration.json")

def circle8():
    return load_array("circle_array8.json", "circle_array8_calibration.json")

def circle4():
    return ArrayState(np.array(
    [
        [0, 0, 0.01],
        [0.01, 0, 0],
        [0, 0, -0.01],
        [-0.01, 0, 0],
    ]).T)

def dipole2():
    return ArrayState(np.array(
    [
        [0, 0, 0.01],
        [0, 0, -0.01],
    ]).T)


In [ ]:
# beamforming utils

def polar_to_unitdir(aziv, zeniv):
    unit_look_direction = np.stack((
        np.cos(aziv) * np.sin(zeniv),
        np.sin(aziv) * np.sin(zeniv),
        np.cos(zeniv)), axis=-1)
    return unit_look_direction

# half-sphere of look directions
def create_look_directions(n, fov):

    fov += 1e-5
    x = np.linspace(-fov, fov, n)
    y = np.linspace(-fov, fov, n)

    xv, yv = np.meshgrid(x, y)

    # circle the square
    # ratio = np.maximum(np.abs(xv), np.abs(yv)) / (np.sqrt(xv*xv + yv*yv) + 1e-18)
    # xv *= ratio
    # yv *= ratio
    
    xv = np.reshape(xv, -1)
    yv = np.reshape(yv, -1)
    aziv = np.atan2(yv, xv)
    zeniv = np.sqrt(xv*xv + yv*yv)

    unit_look_direction = polar_to_unitdir(aziv, zeniv)
    
    # xy = np.stack((xv, yv), axis = -1)

    return (unit_look_direction, aziv, zeniv, (x, y))

def fibonacci_sphere(samples):

    points = []
    phi = np.pi * (np.sqrt(5.) - 1.)  # golden angle in radians

    for i in range(samples):
        y = 1 - (i / float(samples - 1)) * 2  # y goes from 1 to -1
        radius = np.sqrt(1 - y * y)  # radius at y

        theta = phi * i  # golden angle increment

        x = np.cos(theta) * radius
        z = np.sin(theta) * radius

        points.append((x, y, z))

    return np.array(points)

def to_polar_angles(point):
    x, y, z = point
    azi = np.atan2(y, x)
    zeni = np.atan2(np.sqrt(x*x+y*y), z)
    return (azi, zeni)

# look directions (dirs, 3), microphone positions (3, m). returns (dirs, m)
def create_distance_vectors(look_directions, microphone_positions):
    return look_directions @ microphone_positions

def create_steering_vectors(distance_vectors, k):
    return np.exp(1j * distance_vectors * k)

# expects shape (samples, mics)
def resonator_filter(audio, halfTime, fc, sr):
    decay = np.pow(0.5, 1 / (halfTime * sr))
    n = audio.shape[0]
    m = audio.shape[1]
    a = np.full(m, np.exp(2j * np.pi * fc / sr) * decay)
    x = np.zeros(audio.shape,  dtype=np.complex64)
    for i in range(1, n):
        x[i] = x[i-1] * a + audio[i-1] * (1.0 - decay)
    return x

def fb_shuffle(mp):
    m = mp.shape[1]
    nix = np.array([np.argmin(np.linalg.norm(mp + mp[:,i][:,None], axis=0)) for i in range(m)])
    return np.meshgrid(nix, nix)

def analyze_audio(audio, t_idx, nfft, freq, sr, mp, fb=False):
    
    
    m, n = audio.shape
    x = np.zeros(m, dtype=np.complex64)
    R = np.zeros((m, m), dtype=np.complex64)

    if n < 2 * nfft:
        return (x, R)
    
    hann = (0.5 - 0.5 * np.cos(2 * np.pi / nfft * np.arange(nfft))) / nfft
    wavelet = np.exp(-2j * np.pi * freq / sr * np.arange(nfft)) * hann

    pos = max(2 * nfft, t_idx)

    K = 4 * m

    step = nfft // K
    for i in range(K * step, 0, -step):
        x = audio[:,pos-i-nfft:pos-i] @ wavelet
        R += np.conj(np.array([x]).T) @ np.array([x])
    
    # R /= np.sqrt(np.abs(np.sum(np.conj(R)*R)))
    # x /= np.sqrt(np.abs(np.sum(np.conj(x)*x)))

    if fb:
        a, b = fb_shuffle(mp)
        R = (R + np.conj(R[a, b])) * 0.5

    return (x, R)

# def analyze_audio2(audio, t_idx, nfft, freq, sr):
    
    
#     m, n = audio.shape
#     x = np.zeros(m, dtype=np.complex64)
#     R = np.zeros((m, m), dtype=np.complex64)

#     if n < 2 * nfft:
#         return (x, R)
    
#     w1 = np.exp(-2j * np.pi * (freq / sr - 1 / nfft) * np.arange(2*nfft))
#     w2 = np.exp(-2j * np.pi * (freq / sr + 0) * np.arange(2*nfft))
#     w3 = np.exp(-2j * np.pi * (freq / sr + 1 / nfft) * np.arange(2*nfft))
    
#     pos = max(2 * nfft - 1, t_idx)
#     a = audio[:,pos-2*nfft+1:pos+1]
#     s1 = np.cumsum(np.hstack((np.zeros((m, 1)), w1 * a)), axis=-1)
#     s2 = np.cumsum(np.hstack((np.zeros((m, 1)), w2 * a)), axis=-1)
#     s3 = np.cumsum(np.hstack((np.zeros((m, 1)), w3 * a)), axis=-1)

#     step = nfft // (2 * m)
#     for i in range(2 * m * step, 0, -step):
#         x = (- 0.25 * (s1[:,2*nfft-i] - s1[:,nfft-1-i]) 
#              + 0.50 * (s2[:,2*nfft-i] - s2[:,nfft-1-i])
#              - 0.25 * (s3[:,2*nfft-i] - s3[:,nfft-1-i])) / nfft

#         R += np.array([x]).T @ np.array([x])
    
#     R /= 2 * m
#     return (x, R)

In [171]:
# Delay and sum

# x (m, w), steering_vectors (dirs, m)
def das_beamform(x, steering_vectors):
    return np.conj(steering_vectors) @ x / steering_vectors.shape[-1]

def das_beampattern(look_direction, steering_vectors):
    return das_beamform(look_direction, steering_vectors)


In [172]:
# MDF

def distance_steering_matrix(mic_array, k):
    n = mic_array.shape[1]
    dist = np.zeros((n, n), dtype=np.complex128)
    for i in range(n):
        for j in range(n):
            d = mic_array[:,i] - mic_array[:,j]
            dist[i,j] = np.sqrt(np.sum(d*d))
    return np.sinc(k * dist)

def mdf_weights(R, steering_vectors, dial):
    # m = R.shape[0]
    # E, W = np.linalg.eig(R + dial * np.eye(m, dtype=R.dtype))
    # iR = np.zeros((m, m), dtype=R.dtype)
    # ma = np.max(np.abs(E))
    # threshold = ma * 1e-3
    # for i in range(0, m):
    #     if np.abs(E[i]) < threshold:
    #         continue
    #     th = np.array([W[:,i]])
    #     iR += np.conj(th.T) @ th / E[i]
    # w = steering_vectors @ iR
    # norm = np.sum(np.conj(steering_vectors) * w, axis=-1)
    # w /= norm[..., None]
    # return w

    # m = R.shape[0]
    # result = np.zeros(steering_vectors.shape, dtype=np.complex128)
    # iR = np.linalg.inv(R + np.eye(m) * dial)
    # for i in range(steering_vectors.shape[0]):
    #     d = steering_vectors[i,:]
    #     v = iR @ d
    #     result[i,:] = v / (np.conj(d) @ v)

    # return result
    
    m = R.shape[0]
    w = steering_vectors @ np.linalg.inv(R + np.eye(m, dtype=R.dtype) * dial)
    norm = np.sum(np.conj(steering_vectors) * w, axis=-1)
    w /= norm[..., None]

    return w

def mdf_beamform(x, R, steering_vectors, dial):
    w = mdf_weights(R, steering_vectors, dial)
    return np.conj(w) @ x

def mdf_beampattern(look_direction, R, steering_vectors, dial):
    w = mdf_weights(R, np.array([look_direction]), dial)[0]
    return das_beamform(w, steering_vectors)

In [173]:
# MVDR

def mvdr_weights(R, steering_vectors, dial = 1.5):
    loading = dial * np.trace(R).real / R.shape[0]
    w = steering_vectors @ np.linalg.inv(R + loading * np.eye(R.shape[0]))
    norm = np.sum(np.conj(steering_vectors) * w, axis=-1)
    w /= norm[..., None]
    return w

def mvdr_beamform(x, R, steering_vectors, dial = 1.5):
    w = mvdr_weights(R, steering_vectors, dial)
    return np.conj(w) @ x

def mvdr_beampattern(look_direction, R, steering_vectors, dial = 1.5):
    w = mvdr_weights(R, look_direction, dial)
    return steering_vectors @ np.conj(w)

In [174]:
# MUSIC

def music_beamform(R, steering_vectors):
    w = np.linalg.eigh(R)[1][:,-1]
    return np.conj(steering_vectors) @ np.conj(w)

def music_beampattern(R, steering_vectors):
    return music_beamform(R, steering_vectors)

In [175]:
# CroPaC for a planar circular array

def planar_directions(directions):
    return np.atan2(directions[:,1], directions[:,0])

def zeni_directions(directions):
    return np.atan2(np.sqrt(directions[:,0]**2 + directions[:,1]**2), directions[:,2])

def create_cropac_weights(look_direction, look_vector, mic_array, k):

    mic = planar_directions(mic_array.T)
    dir = planar_directions(look_direction[None,:])
    relative_angles = dir[0] - mic

    m = mic_array.shape[1]
    r = np.sqrt(np.sum(mic_array[:,0] * mic_array[:,0]))

    o1 = 1
    o2 = 2

    dj_zeros = [0, 1.8411837813, 3.0542369282, 4.2011889412]

    w1_zeni = np.arcsin(min(1, dj_zeros[o1] / (k * r)))
    w2_zeni = np.arcsin(min(1, dj_zeros[o2] / (k * r)))
    look_zeni = zeni_directions(look_direction[None,:])[0]

    zeni_steer = min(1, k * r / dj_zeros[o1])**2
    zeni_steer1 = k * r * (np.sin(w1_zeni) - np.sin(look_zeni)) * zeni_steer
    zeni_steer2 = k * r * (np.sin(w2_zeni) - np.sin(look_zeni)) * zeni_steer

    kr1z = dj_zeros[o1] - zeni_steer1
    kr2z = dj_zeros[o2] - zeni_steer2
    zeni_steer2 += (kr2z - kr1z) * zeni_steer

    kr1n = min(k * r + zeni_steer1, dj_zeros[o1])
    kr2n = min(k * r + zeni_steer2, dj_zeros[o2])

    w1 = np.cos(o1 * relative_angles) * (1j**o1) / (jv(o1, kr1n)) / m
    w2 = np.cos(o2 * relative_angles) * (1j**o2) / (jv(o2, kr2n)) / m

    dv0 = create_distance_vectors(polar_to_unitdir(dir, np.array([np.pi/2])), mic_array)

    sv1 = create_steering_vectors(dv0, zeni_steer1 / r)
    sv2 = create_steering_vectors(dv0, zeni_steer2 / r)

    correction1 = np.conj(sv1)
    correction2 = np.conj(sv2)

    w1 = w1 * correction1[0]
    w2 = w2 * correction2[0]

    das_w1 = look_vector / m * 2
    das_w2 = look_vector / m * 2

    for i in range(8):
        if i&1:
            das_w2[i] = 0
        else:
            das_w1[i] = 0

    j0_zero = 2.4048
    mlw = np.arcsin(min(1, j0_zero / (k*r)))

    bstart = 45 / todeg
    bend = 30 / todeg
    mlw = min(bstart, max(mlw, bend))

    blend = (mlw - bend) / (bstart - bend)

    w1 = w1 * blend + das_w1 * (1 - blend)
    w2 = w2 * blend + das_w2 * (1 - blend)

    return w1, w2

def cropac_beamform(look_directions, steering_vectors, mic_array, k, x, tp):
    
    m = steering_vectors.shape[1]
    w1 = np.zeros_like(steering_vectors)
    w2 = np.zeros_like(steering_vectors)

    for i in range(steering_vectors.shape[0]):
        ww1, ww2 = create_cropac_weights(look_directions[i], steering_vectors[i], mic_array, k)
        w1[i,:] = ww1
        w2[i,:] = ww2

    z1 = np.conj(w1) @ x
    z2 = np.conj(w2) @ x
    das = np.conj(steering_vectors) @ x / m

    # norm = np.sum(np.abs(x)**2) / m
    norm = ((np.abs(z1)**2 + np.abs(z2)**2)) / 2

    if tp == 1:
        return z1
    if tp == 2:
        return z2
    return das * np.maximum(0, np.real(np.conj(z1) * z2) / norm)

def cropac_beampattern(look_direction, look_vector, steering_vectors, mic_array, k, tp):

    w1, w2 = create_cropac_weights(look_direction, look_vector, mic_array, k)

    m = steering_vectors.shape[1]
    z1 = steering_vectors @ np.conj(w1)
    z2 = steering_vectors @ np.conj(w2)
    das = steering_vectors @ np.conj(look_vector) / m

    norm = ((np.abs(z1)**2 + np.abs(z2)**2)) / 2
    # norm = 1

    if tp == 1:
        return z1
    if tp == 2:
        return z2
    return das * np.maximum(0, np.real(np.conj(z1) * z2)) / norm

In [176]:
# adaptive CroPaC

def create_primary_weights(look_direction, look_vector, mic_array, k):

    mic = planar_directions(mic_array.T)
    dir = planar_directions(look_direction[None,:])
    relative_angles = dir[0] - mic

    m = mic_array.shape[1]
    r = np.sqrt(np.sum(mic_array[:,0] * mic_array[:,0]))

    o1 = 1

    dj_zeros = [0, 1.8411837813, 3.0542369282, 4.2011889412]

    w1_zeni = np.arcsin(min(1, dj_zeros[o1] / (k * r)))
    look_zeni = zeni_directions(look_direction[None,:])[0]

    zeni_steer = min(1, k * r / dj_zeros[o1])**2
    zeni_steer1 = k * r * (np.sin(w1_zeni) - np.sin(look_zeni)) * zeni_steer
    kr1n = min(k * r + zeni_steer1, dj_zeros[o1])
    w1 = np.cos(o1 * relative_angles) * (1j**o1) / (jv(o1, kr1n)) / m

    dv0 = create_distance_vectors(polar_to_unitdir(dir, np.array([np.pi/2])), mic_array)

    sv1 = create_steering_vectors(dv0, zeni_steer1 / r)

    correction1 = np.conj(sv1)

    w1 = w1 * correction1[0]

    das_w1 = look_vector / m * 2

    for i in range(8):
        if i&1:
            das_w1[i] = 0

    j0_zero = 2.4048
    mlw = np.arcsin(min(1, j0_zero / (k*r)))

    bstart = 45 / todeg
    bend = 30 / todeg
    mlw = min(bstart, max(mlw, bend))

    blend = (bstart - mlw) / (bstart - bend)

    w1 = w1 * (1 - blend) + das_w1 * blend

    return w1

def create_optimal_secondary_weights(w1, look_vector, constraints):
    
    N = w1.shape[0]
    nsdim = N - len(constraints)
    assert nsdim > 0

    constraints = [i / np.linalg.norm(i) for i in constraints]
    for i in range(N-nsdim):
        for j in range(0, i):
            constraints[i] -= (constraints[i] @ np.conj(constraints[j])) * constraints[j]
            constraints[i] /= np.linalg.norm(constraints[i])

    O = []
    for i in range(nsdim):
        basisv = np.random.randn(N).astype(np.complex128)
        basisv /= np.linalg.norm(basisv)
        for v in constraints:
            basisv -= (basisv @ np.conj(v)) * v
            basisv /= np.linalg.norm(basisv)
        constraints.append(basisv)
        O.append(basisv)

    O = np.array(O).T

    # w2 = O @ np.linalg.inv(np.conj(O).T @ O) @ np.conj(O).T @ look_vector
    w2 = O @ np.linalg.pinv(O) @ look_vector
    w2 = w2 / (np.conj(w2) @ look_vector)

    return w2

def opt_beamform(look_directions, steering_vectors, mic_array, constraints, k, x,):
    
    m = steering_vectors.shape[1]
    w1 = np.zeros_like(steering_vectors)
    w2 = np.zeros_like(steering_vectors)

    for i in range(steering_vectors.shape[0]):
        ww1 = create_primary_weights(look_directions[i], steering_vectors[i], mic_array, k)
        ww2 = create_optimal_secondary_weights(ww1, steering_vectors[i], [S @ ww1 for S in constraints])
        w1[i,:] = ww1
        w2[i,:] = ww2

    z1 = np.conj(w1) @ x
    z2 = np.conj(w2) @ x

    return np.maximum(0, np.real(np.conj(z1) * z2))

def opt_beampattern(look_direction, look_vector, steering_vectors, mic_array, constraints, k):

    w1 = create_primary_weights(look_direction, look_vector, mic_array, k)
    w2 = create_optimal_secondary_weights(w1, look_vector, [S @ w1 for S in constraints])

    z1 = steering_vectors @ np.conj(w1)
    z2 = steering_vectors @ np.conj(w2)

    return np.maximum(0, np.real(np.conj(z1) * z2))

In [177]:
# Static tests

def create_pickup_pattern(V, t, b):
    return np.linalg.inv(np.conj(V).T @ V + b * np.eye(V.shape[1])) @ np.conj(V).T @ t

def dipole_beampattern(look_directions, mp, k):
    
    dk = look_directions * k
    phi = planar_directions(look_directions)
    r = np.sqrt(np.sum(mp[:,0] * mp[:,0]))

    kr = np.sqrt(dk[:,0]**2 + dk[:,1]**2)

    return jv(1, kr) * np.cos(phi)

def quadropole_beampattern(look_directions, mp, k):

    dk = look_directions * k
    phi = planar_directions(look_directions)
    r = np.sqrt(np.sum(mp[:,0] * mp[:,0]))

    kr = np.sqrt(dk[0]**2 + dk[1]**2)
    

    return jv(2, kr) * np.cos(2 * phi)

def random_beampattern(steering_vectors):
    rnd = np.random.rand(steering_vectors.shape[1]) * 2 - 1
    return steering_vectors @ rnd

In [178]:
# plot utils

def plot_3d_base(ax, azim, elev, size=7.5):
    """
    Plot a 3D flight path and a reference point at the origin.
    points: numpy array of shape (3, N)
    t: numpy array of shape (N,)
    """
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.view_init(elev=elev, azim=azim)

    ax.grid(True)

    ax.set_xlim(-size, size)
    ax.set_ylim(-size, size)
    ax.set_zlim(0, 2 * size)

def plot_path_3d(ax, points, t_idx, color, marker='o'):
    resolution = 2**12
    step = max(1, points.shape[1] // resolution)
    idx = np.arange(0, points.shape[1], step)
    if points.shape[1] % step != 0:
        idx = np.append(idx, [points.shape[1]-1]) 
    p = points[:,idx]
    x, y, z = p

    ax.plot(x, y, z, color=color)
    ax.plot(*points[:,t_idx], color=color, markersize=10, linestyle='', marker=marker)

def plot_position_3d(ax, position, t_idx, color, marker='o'):
    if isinstance(position, tuple):
        plot_path_3d(ax, position[0], t_idx, color, marker)
    else:
        ax.plot(*position, color=color, markersize=10, linestyle='', marker=marker)

def plot_mesh_polar(ax, plot_points, z):
    aziv, zeniv = plot_points
    c = ax.pcolormesh(aziv, zeniv * todeg, np.abs(z), shading='gouraud', cmap='viridis')
    ax.set_ylim([0, 90 + 1e-5])
    plt.colorbar(c, ax=ax)
    # plt.colorbar(c, ax=ax, format=ticker.StrMethodFormatter('{x:.2f}'))

In [ ]:
c = 344
total_duration = 10
fig_width = 10
dpi = 80
sr = 48000
beamforming_resolution = 64
fov = 90 / todeg

look_directions, aziv, zeniv, xy = create_look_directions(beamforming_resolution, fov)
plot_points = (aziv.reshape((beamforming_resolution, beamforming_resolution)),
               zeniv.reshape((beamforming_resolution, beamforming_resolution)))

rnd = np.random.randint(0, beamforming_resolution**2, 10)

# for i in rnd:
#     print(xy[0][i%beamforming_resolution], xy[1][i//beamforming_resolution], aziv[i], zeniv[i], look_directions[i])


microphone_arrays = {
    "octagon": circle8,
    "square8_2": square8_2,
    "default8_1": default8_1,
    "default8_2": default8_2,
    "default8_3": default8_3,
    "square4": square4,
    "square8": square8,
    "cube3": cube3,
    "random8d01": random8d01,
    "random16d01": random16d01,
    "linear": linear8,
    "square9": square9,
    "dipole": dipole2,
    "quad": circle4,
    "hexagon": hexagon
}

total_samples = total_duration * sr

audio_sources = {
    "noise": (np.random.randn(total_samples), sr),
    "mavic4":  "../samples/mavic4_1s.wav",
}

scenes = {
    "zeni": scene_zeni,
    "circle2": scene_circle2,
    "stationary": scene_stationary,
    "center": scene_center,
    "line": scene_line,
    "circle": scene_circle,
    "helix": scene_helix,
    "random": scene_random,
    "interferer": scene_interferer,
}

class State:
    scene: Scene = None
    mic_array = ArrayState()
    perfect_array = np.zeros((1, 1), dtype=np.float32)
    distance_vectors = np.zeros((1, 1), dtype=np.float32)
    target = np.zeros((1, 1), dtype=np.float32)
    selfnoise = np.zeros((1, 1), dtype=np.float32)
    diffuse = np.zeros((1, 1), dtype=np.float32)
    interferer = np.zeros((1, 1), dtype=np.float32)
    noise = np.zeros((1, 1), dtype=np.float32)
    audio = np.zeros((1, 1), dtype=np.float32)
    
state = State()

w_array = Dropdown(options=[*microphone_arrays.keys()], value=[*microphone_arrays.keys()][0], description="Array:")
w_target = Dropdown(options=[*audio_sources.keys()], value=[*audio_sources.keys()][0], description="Target:")
w_scene = Dropdown(options=[*scenes.keys()], value=[*scenes.keys()][0], description="Scene:")
w_NFFT = Dropdown(options=[2**i for i in range(6, 14)], value=1024, description="NFFT:")
w_algo = Dropdown(options=["DaS", "MVDR", "OPT", "MUSIC", "MDF", "CroPaC", "dipole", "random"], value="OPT", description="Algorithm:")
w_covar = Dropdown(options=["Avg", "FB"], value="FB", description="Cx:")
for i in [w_array, w_target, w_scene, w_NFFT, w_algo]:
    i.layout.width = f"{180}px"

w_time = FloatSlider(
    value=0.0, min=0.0, max=max(0.0, total_duration - 0.01),
    step=max(0.001, total_duration / 1000.0),
    description="Time (s)", continuous_update=True, readout_format=".2f"
)
w_azi = FloatSlider(
    value=-60, min=-180, max=180, step=1,
    description="Azi (deg)", continuous_update=True, readout_format=".0f"
)
w_ele = FloatSlider(
    value=30, min=-90, max=90, step=1,
    description="Ele (deg)", continuous_update=True, readout_format=".0f"
)



w_position_error = FloatSlider(
    value=0, min=0, max=20, step=0.1,
    description="Pos (mm)", continuous_update=False, readout_format=".1f"
)
w_level_error = FloatSlider(
    value=0, min=0, max=10, step=0.1,
    description="Level (dB)", continuous_update=True, readout_format=".1f"
)
w_phase_error = FloatSlider(
    value=0, min=0, max=1, step=0.01,
    description="Phase (rad)", continuous_update=True, readout_format=".2f"
)
w_regularization = FloatSlider(
    value=-80, min=-80, max=30, step=0.01,
    description="Beta (dB)", continuous_update=True, readout_format=".2f"
)
for i in [w_position_error, w_level_error, w_phase_error, w_regularization]:
    i.layout.width = f"{fig_width * dpi * 0.6}px"



w_noise_db = FloatSlider(
    value=-80, min=-80, max=20, step=0.1,
    description="Noise (dB)", continuous_update=True, readout_format=".0f"
)
w_diffuse_db = FloatSlider(
    value=-80, min=-80, max=20, step=0.1,
    description="Diffuse (dB)", continuous_update=True, readout_format=".0f"
)
w_interferer_db = FloatSlider(
    value=-80, min=-80, max=20, step=0.1,
    description="Interf. (dB)", continuous_update=True, readout_format=".0f"
)
w_target_db = FloatSlider(
    value=0, min=-80, max=20, step=0.1,
    description="Target (dB)", continuous_update=True, readout_format=".0f"
)
w_freq = FloatSlider(
    value=1000, min=100, max=4000, step=1,
    description="Freq (Hz)", continuous_update=True, readout_format=".0f"
)
for i in [w_time, w_azi, w_ele, w_noise_db, w_diffuse_db, w_interferer_db, w_target_db, w_freq]:
    i.layout.width = f"{fig_width * dpi * 0.6}px"

# look directions

out1 = Output()
out2 = Output()

def update_plot(change=None):
    time = max(0.0, min(w_time.value, total_duration))
    t_idx = int(round(time * sr))
    t_idx = max(0, min(t_idx, total_samples-1))

    mp = state.mic_array.positions
    dv = state.distance_vectors
    perr = w_phase_error.value

    k = 2 * np.pi * w_freq.value / c
    steering_vectors = create_steering_vectors(dv, k)

    z_view = np.zeros(beamforming_resolution * beamforming_resolution)
    z_pickup = np.zeros(beamforming_resolution * beamforming_resolution)

    target_direction = get_position(state.scene.target_position, t_idx)
    unit_target_direction = target_direction / (np.sqrt(np.sum(target_direction**2)) + 1e-18)
    target_polar = np.array(to_polar_angles(target_direction))
    target_polar[1] *= todeg

    interferer_polar = []
    for interferer in state.scene.interferer_position:
        pos = get_position(interferer, t_idx)
        pol = np.array(to_polar_angles(pos))
        pol[1] *= todeg
        interferer_polar.append(pol)

    with out1:
        clear_output(wait=True)

        x, R = analyze_audio(state.audio, t_idx, int(w_NFFT.value), w_freq.value, sr, mp, w_covar.value=="FB")
        phe = np.exp(perr * np.random.randn(mp.shape[1]) * 1j)
        x *= phe
        R *= np.conj(np.array([phe]).T) @ np.array([phe])

        v0 = create_steering_vectors(create_distance_vectors(np.array([unit_target_direction]), mp), k)
        reg = 10 ** (w_regularization.value / 20)
        if np.abs(w_regularization.value + 80) < 0.01:
            reg = 0

        if w_algo.value == "DaS":
            z_view = das_beamform(x, steering_vectors)
            z_pickup = das_beampattern(v0[0], steering_vectors)
        elif w_algo.value == "MVDR":
            z_view = mvdr_beamform(x, R, steering_vectors, reg)
            z_pickup = mvdr_beampattern(v0[0], R, steering_vectors, reg)
        elif w_algo.value == "MUSIC":
            z_view = music_beamform(R, steering_vectors)
            z_pickup = music_beampattern(R, steering_vectors)
        elif w_algo.value == "MDF":
            Rd = distance_steering_matrix(mp, k)
            z_view = mdf_beamform(x, Rd, steering_vectors, reg)
            z_pickup = mdf_beampattern(v0[0], Rd, steering_vectors, reg)
        elif w_algo.value == "dipole":
            z_pickup = dipole_beampattern(look_directions, mp, k)
            z_view = quadropole_beampattern(look_directions, mp, k)
        elif w_algo.value == "CroPaC":
            tp = 1
            if w_regularization.value > -60:
                tp = 2
            if w_regularization.value > -10:
                tp = 3
            z_view = cropac_beamform(look_directions, steering_vectors, state.perfect_array, k, x, tp)
            z_pickup = cropac_beampattern(unit_target_direction, v0[0], steering_vectors, state.perfect_array, k, tp)
        elif w_algo.value == "random":
            z_pickup = random_beampattern(steering_vectors)
            z_pickup = z_pickup / np.max(np.abs(z_pickup))
        elif w_algo.value == "OPT":
            loading = reg * np.trace(R).real / R.shape[0]
            constraints = [R + loading * np.eye(R.shape[0])]
            z_view = opt_beamform(look_directions, steering_vectors, state.perfect_array, constraints, k, x)
            z_pickup = opt_beampattern(unit_target_direction, v0[0], steering_vectors, state.perfect_array, constraints, k)

        z_view = z_view.reshape((beamforming_resolution, beamforming_resolution))
        z_pickup = z_pickup.reshape((beamforming_resolution, beamforming_resolution))

        fig = plt.figure(figsize=(fig_width*0.5, fig_width*0.35))
        ax = fig.add_subplot(111, projection='3d')
        plot_3d_base(ax, w_azi.value, w_ele.value)
        ax.plot(0,0,0 , color='red', markersize=4, linestyle='', marker='o')
        ax.plot(*(mp*10) , color='red', markersize=1, linestyle='', marker='o')
        # plt.gca().tick_params(labelbottom=False, labelleft=False)
        
        plot_position_3d(ax, state.scene.target_position, t_idx, 'blue')
        for interferer in state.scene.interferer_position:
            plot_position_3d(ax, interferer, t_idx, 'C1', marker='x')
        plt.tight_layout()
        plt.show()

    with out2:
        clear_output(wait=True)
        fig = plt.figure(figsize=(fig_width, fig_width*0.4))

        ax_view = fig.add_subplot(121, projection='polar')
        plot_mesh_polar(ax_view, plot_points, z_view)
        ax_view.scatter(*target_polar, color='blue', s=100)
        for i in interferer_polar:
            ax_view.scatter(*i, color='C1', marker="x", s=100)
        ax_view.set_title("Acoustic view")

        ax_pickup = fig.add_subplot(122, projection='polar')
        plot_mesh_polar(ax_pickup, plot_points, z_pickup)
        for i in interferer_polar:
            ax_pickup.scatter(*i, color='C1', marker="x", s=100)
        ax_pickup.scatter(*target_polar, color='blue', s=100)
        ax_pickup.set_title("Pickup pattern")
        
        plt.tight_layout()
        plt.show()

        display(Audio(state.audio[0], rate=sr))

def update_audio(change=None, update_plt=True):
    g_n = lin_scale(w_noise_db.value)
    g_t = lin_scale(w_target_db.value)
    g_i = lin_scale(w_interferer_db.value)
    g_d = lin_scale(w_diffuse_db.value)
    state.noise = state.selfnoise * g_n + state.interferer * g_i + state.diffuse * g_d
    state.audio = state.target * g_t + state.noise
    level_error = 10**(np.random.randn(state.audio.shape[0]) * w_level_error.value / 20)
    state.audio *= level_error[...,None]
    if update_plt:
        update_plot()

def update_scene(change=None, update_plt=True):
    scene = scenes[w_scene.value](total_duration, sr)
    state.scene = scene
    m = state.mic_array.positions.shape[1]

    state.target = prepare_audio(
        audio_sources[w_target.value],
        state.mic_array.positions,
        scene.target_position, total_samples, sr)
    
    state.target = rms_normalize(state.target)
    
    state.interferer = np.zeros((m, total_samples), dtype=np.float32)
    for i in range(len(scene.interferer_audio)):
        state.interferer += prepare_audio(
            scene.interferer_audio[i], 
            state.mic_array.positions, 
            scene.interferer_position[i], total_samples, sr)
        
    state.interferer = rms_normalize(state.interferer)
    
    update_audio(update_plt=update_plt)

def update_mic_array(change=None, update_plt=True):
    mic_array = microphone_arrays[w_array.value]()
    state.mic_array = mic_array
    state.perfect_array = np.copy(mic_array.positions)
    m = mic_array.positions.shape[1]
    state.distance_vectors = create_distance_vectors(look_directions, state.perfect_array)
    error = np.random.randn(3, m) * w_position_error.value * 0.001
    error[2] *= 0.5
    mic_array.positions += error
    state.selfnoise = create_uncorrelated_noise(m, total_samples)
    state.diffuse = create_diffuse_noise(total_samples, mic_array.positions, sr)
    update_scene(update_plt=update_plt)

w_array.observe(update_mic_array, names="value")
w_position_error.observe(update_mic_array, names="value")

for w in (w_target, w_scene):
    w.observe(update_scene, names="value")

for w in (w_time, w_azi, w_ele, w_NFFT, w_algo, w_covar):
    w.observe(update_plot, names="value")

for w in (w_target_db, w_noise_db, w_diffuse_db, w_interferer_db, w_freq, w_level_error, w_phase_error, w_regularization):
    w.observe(update_audio, names="value")

display(VBox([
    HBox([w_array, w_target, w_scene, w_algo, w_NFFT, w_covar]),
    HBox([
        VBox([w_time, w_ele, w_azi,
              Label("Array errors and tolerances"),
              w_position_error, w_level_error, w_phase_error, w_regularization,
              Label("Noise levels"),
              w_target_db, w_noise_db, w_interferer_db, w_diffuse_db, w_freq
              ]),
        out1
    ]),
    out2
]))

update_mic_array()

In [180]:
a = np.ones((12, 4))
print(a / np.sum(a, axis=-1)[..., None])

a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30])
result = a[None, :] - b[:, None]
# print(result)

a = np.array([1300, 6000])
k = 2 * np.pi * a / c
print(k)

print(c / a / 2)


[[0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]]
[ 23.74459564 109.5904414 ]
[0.13230769 0.02866667]
